In [1]:
# -*- coding: utf-8 -*-

Zaimplementuj aplikację szacującą czas ukończenia półmaratonu dla zadanych danych

1. Umieść dane w Digital Ocean Spaces

1. Napisz notebook, który będzie Twoim pipelinem do trenowania modelu
    * czyta dane z Digital Ocean Spaces
    * czyści je
    * trenuje model (dobierz odpowiednie metryki [feature selection])
    * nowa wersja modelu jest zapisywana lokalnie i do Digital Ocean Spaces

1. Aplikacja
    * opakuj model w aplikację streamlit
    * wdróż (deploy) aplikację za pomocą Digital Ocean AppPlatform 
    * wejściem jest pole tekstowe, w którym użytkownik się przedstawia, mówi o tym
    jaka jest jego płeć, wiek i czas na 5km
    * jeśli użytkownik podał za mało danych, wyświetl informację o tym jakich danych brakuje
    * za pomocą LLM (OpenAI) wyłuskaj potrzebne dane, potrzebne dla Twojego modelu
    do określenia, do słownika (dictionary lub JSON)
    * tę część podepnij do Langfuse, aby zbierać metryki o skuteczności działania LLM'a



In [2]:
import sys
import io
import os
import getpass
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.io as pio
import csv
from faker import Faker
from pycaret.classification import setup, pull
import re
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.metrics import homogeneity_score, completeness_score,accuracy_score, recall_score, precision_score, f1_score, roc_auc_score
from pycaret.clustering import *
from dotenv import set_key, load_dotenv
from itables import show
import boto3


# Poprawiony import Pandera
import pandera as pa
from pandera import Column, DataFrameSchema, Check

# Import PyCaret
from pycaret.clustering import setup, create_model, save_model, plot_model, assign_model, predict_model, pull

# Ustawienie ścieżki
PROJECT_ROOT = Path.cwd().parent
# Ścieżka do podkatalogu data
data_dir = PROJECT_ROOT / "data"

# Jeśli chcesz stworzyć ten folder, gdyby nie istniał:
data_dir.mkdir(exist_ok=True)




# Funkcje odpowiedzialne za komunikację z digital ocean i openai

Funkcje próbują wczytać klucze z pliku env, a w przypadku braku pokazują pole tekstowe do wprowadzenia danych. Dalej mamy też funkcję wysyłającą dane na serwer digital ocean.

In [3]:
def handle_digital_ocean_keys():
    # 1. Próba załadowania z .env
    current_folder = Path.cwd().parent
    env_path = current_folder / ".env"
    load_dotenv(env_path)
    #print(env_path)
    # Pobranie wartości z systemu/pliku .env
    do_access_key = os.getenv("DO_ACCESS_KEY")
    do_secret_key = os.getenv("DO_SECRET_KEY")
    do_space_input = os.getenv("DO_SPACE_NAME")
    do_region = os.getenv("DO_REGION")
    do_end_url = os.getenv("DO_END_URL")
    
    # Sprawdzamy, czy klucze już są załadowane
    keys_loaded = all([do_access_key, do_secret_key, do_space_input])

    if not keys_loaded:
        print("Konfiguracja DigitalOcean Spaces")
        do_access_key = getpass.getpass("Wprowadź DigitalOcean Access Key: ")
        do_secret_key = getpass.getpass("Wprowadź secret digital oceans space Key")
        do_space_input = input("Wprowadź nazwę Space (Bucketa): ")
        do_region = input("Wprowadź region (np. fra1): ")
        do_end_url = input("Wprowadź url region (np. https://fra1.digitaloceanspaces.com): ")
                
        if do_access_key and do_secret_key and do_space_input:
            # Ustawienie zmiennych środowiskowych, aby os.getenv je widział po rerun
            os.environ["DO_ACCESS_KEY"] = do_access_key
            os.environ["DO_SECRET_KEY"] = do_secret_key
            os.environ["DO_SPACE_NAME"] = do_space_input
            os.environ["DO_REGION"] = do_region
            os.environ["DO_END_URL"] = do_end_url
            set_key(str(env_path), "DO_ACCESS_KEY", do_access_key)
            set_key(str(env_path), "DO_SECRET_KEY", do_secret_key)
            set_key(str(env_path), "DO_SPACE_NAME", do_space_input)
            set_key(str(env_path), "DO_REGION", do_region)
            set_key(str(env_path), "DO_END_URL", do_end_url)
            print("Klucze zapisane!")
        else:
            print("Wypełnij wszystkie pola!")
            
    return do_access_key, do_secret_key, do_space_input, do_region, do_end_url

In [4]:
def digital_ocean_to_csv(access_key,access_secret_key, nazwa_space,plik_csv,region = 'fra1',end_url='https://fra1.digitaloceanspaces.com'):
    load_dotenv()

    # Konfiguracja sesji
    session = boto3.session.Session()
    client = session.client(
        's3',
        region_name=region.strip(),
        endpoint_url=end_url.strip(), # Używamy endpointu bazowego do komunikacji API
        aws_access_key_id=access_key.strip(),
        aws_secret_access_key=access_secret_key.strip()
    )

    # Pobieranie pliku bezpośrednio do pamięci (bez zapisu na dysku)
    response = client.get_object(Bucket=nazwa_space, Key=plik_csv)
    csv_data = response['Body'].read()

    # Wczytanie do Pandas
    return pd.read_csv(io.BytesIO(csv_data), sep=';',encoding='utf-8-sig')

In [5]:
def upload_model_to_digital_ocean(local_file_path, access_key, access_secret_key, nazwa_space, remote_path=None, region='fra1', end_url='https://fra1.digitaloceanspaces.com'):
    # Jeśli nie podano nazwy w chmurze, użyj nazwy pliku lokalnego
    if remote_path is None:
        remote_path = os.path.basename(local_file_path)
        
    session = boto3.session.Session()
    client = session.client(
        's3',
        region_name=region.strip(),
        endpoint_url=end_url.strip(),
        aws_access_key_id=access_key.strip(),
        aws_secret_access_key=access_secret_key.strip()
    )

    try:
        # Wysyłanie pliku binarnego .pkl
        client.upload_file(local_file_path, nazwa_space, remote_path)
        print(f"Model {local_file_path} został wysłany do Space: {nazwa_space} jako {remote_path}")
    except Exception as e:
        print(f"Błąd wysyłki modelu: {e}")


# Funkcje pomocnicze

Funkcje pomocnicze odpowiedzialne za przeprocesowanie datasetu do uczenia maszynowego oraz do konwersji czasu z formatu time na ilość sekund

In [6]:
# Snippet pomocniczy - zmiana czasu na sekundy

def convert_time_to_seconds(time):
      # Jeśli wartość to NaN (float) lub nie jest stringiem, zwróć None lub 0
    if not isinstance(time, str):
        return 0  # lub return 0
    if pd.isnull(time) or time in ['DNS', 'DNF']:
        return 0
    time = time.split(':')
    return int(time[0]) * 3600 + int(time[1]) * 60 + int(time[2])

In [7]:
def bezpieczna_dominanta(seria, wartosc_domyslna):
    m = seria.mode()
    return m.iloc[0] if not m.empty else wartosc_domyslna

# Wczytanie danych z digital ocean

Wczytujemy wcześniej zapisane dane z digital ocean oraz wyświetlamy ilość wczytanych wierszy, by mieć pewność poprawności wczytania.

In [8]:
#df_maraton23 = pd.read_csv(data_dir / "halfmarathon_wroclaw_2023__final.csv", encoding='utf-8-sig',sep=';')
do_access_key, do_secret_key, do_space_input, do_region, do_end_url = handle_digital_ocean_keys()
df_maraton23 = digital_ocean_to_csv(do_access_key, do_secret_key, do_space_input, "halfmarathon_wroclaw_2023__final.csv",do_region, do_end_url)
df_maraton23.columns = df_maraton23.columns.str.strip()
print(f"Pomyślnie wczytano dane. Liczba wierszy: {len(df_maraton23)}")

Pomyślnie wczytano dane. Liczba wierszy: 8950


In [9]:
#df_maraton24 = pd.read_csv(data_dir / "halfmarathon_wroclaw_2024__final.csv", encoding='utf-8-sig', sep=';')
do_access_key, do_secret_key, do_space_input, do_region, do_end_url = handle_digital_ocean_keys()
df_maraton24 = digital_ocean_to_csv(do_access_key, do_secret_key, do_space_input, 'halfmarathon_wroclaw_2024__final.csv',do_region, do_end_url)
df_maraton24.columns = df_maraton23.columns.str.strip()
print(f"Pomyślnie wczytano dane. Liczba wierszy: {len(df_maraton24)}")

Pomyślnie wczytano dane. Liczba wierszy: 13007


# Wypełanianie brakujących danych

Rzutujemy poszczególne kolumny na dany typ i wypełniamy brakujące dane. Kolumny liczbowe wypełniamy zerami. Poszczególne odcinki czasowe zamieniamy z formatu time na ilość sekund w celu ułatwienia dalszej analizy
 

In [10]:
df_maraton23['Miejsce'] = pd.to_numeric(df_maraton23['Miejsce'], errors='coerce').fillna(0).astype('Int64')
df_maraton23['Płeć Miejsce'] = pd.to_numeric(df_maraton23['Płeć Miejsce'], errors='coerce').fillna(0).astype('Int64')
df_maraton23['Kategoria wiekowa Miejsce'] = pd.to_numeric(df_maraton23['Kategoria wiekowa Miejsce'], errors='coerce').fillna(0).astype('Int64')
df_maraton23['Rocznik'] = pd.to_numeric(df_maraton23['Rocznik'], errors='coerce').fillna(0).astype('Int64')
df_maraton23['Czas'] = df_maraton23['Czas'].apply(convert_time_to_seconds)
df_maraton23['5 km Czas'] = df_maraton23['5 km Czas'].apply(convert_time_to_seconds)
df_maraton23['10 km Czas'] = df_maraton23['10 km Czas'].apply(convert_time_to_seconds)
df_maraton23['15 km Czas'] = df_maraton23['15 km Czas'].apply(convert_time_to_seconds) 
df_maraton23['20 km Czas'] = df_maraton23['20 km Czas'].apply(convert_time_to_seconds) 


In [11]:
df_maraton23.sample(10)

,Miejsce,Numer startowy,Imię,Nazwisko,Miasto,Kraj,Drużyna,Płeć,Płeć Miejsce,Kategoria wiekowa,...,10 km Tempo,15 km Czas,15 km Miejsce Open,15 km Tempo,20 km Czas,20 km Miejsce Open,20 km Tempo,Tempo Stabilność,Czas,Tempo
4586,4587,3584,DAWID,PALUSZCZAK,WROCŁAW,POL,TOWARZYSZE,M,3789,M20,...,5.763333,5306,5368.0,5.863333,7085,4703.0,5.930000,-0.005800,7386,5.835506
5857,5858,2035,MARCJANNA,BACIK,OSTRZESZÓW,POL,NaN,K,1240,K20,...,5.860000,5368,5558.0,6.210000,7544,5796.0,7.253333,0.092800,7946,6.277949
4796,4797,4896,WOJCIECH,GOŁĘBIOWSKI,WROCŁAW,POL,NaN,M,3940,M20,...,5.500000,5284,5307.0,6.656667,7116,4794.0,6.106667,0.062133,7467,5.899502
7783,7784,4285,PAULINA,UJMA,WYSOKA,POL,NaN,K,2149,K30,...,6.863333,6466,7753.0,7.620000,9196,7800.0,9.100000,0.136933,9632,7.610018
4463,4464,4273,HONORATA,ULATOWSKA,SZCZODRE,POL,NaN,K,760,K40,...,5.603333,5047,4363.0,5.706667,6987,4461.0,6.466667,0.059267,7337,5.796792
6228,6229,5086,ELŻBIETA,KURIATA,KUKLICE,POL,NaN,K,1399,K40,...,6.150000,5662,6399.0,6.283333,7759,6237.0,6.990000,0.035667,8142,6.432804
2202,2203,1424,ARTUR,ZYTKO,SIECHNICE,POL,NaN,M,1989,M50,...,5.016667,4603,2430.0,5.203333,6270,2242.0,5.556667,0.029733,6559,5.182113
8808,0,5134,RADOSŁAW,SZEWC,NaN,NaN,NaN,M,0,M50,...,NaN,0,NaN,NaN,0,NaN,NaN,NaN,0,NaN
1864,1865,5189,JAN,LECHOWSKI,JELCZ-LASKOWICE,POL,KB HARCOWNIK JELCZ-LASKOWICE,M,1703,M60,...,4.796667,4457,1872.0,5.140000,6123,1843.0,5.553333,0.044867,6434,5.083353
6291,6292,5474,MICHALINA,PIETRUSZEWSKA,KARWIANY,POL,NaN,K,1420,K30,...,5.763333,5339,5450.0,6.250000,7721,6179.0,7.940000,0.139133,8190,6.470728


In [13]:
df_maraton24['Miejsce'] = pd.to_numeric(df_maraton24['Miejsce'], errors='coerce').fillna(0).astype('int64')
df_maraton24['Płeć Miejsce'] = pd.to_numeric(df_maraton24['Płeć Miejsce'], errors='coerce').fillna(0).astype('int64')
df_maraton24['Kategoria wiekowa Miejsce'] = pd.to_numeric(df_maraton24['Kategoria wiekowa Miejsce'], errors='coerce').fillna(0).astype('int64')
df_maraton24['Rocznik'] = pd.to_numeric(df_maraton24['Rocznik'], errors='coerce').fillna(0).astype('int64')
df_maraton24['Czas'] = df_maraton24['Czas'].apply(convert_time_to_seconds)
df_maraton24['5 km Czas'] = df_maraton24['5 km Czas'].apply(convert_time_to_seconds)
df_maraton24['10 km Czas'] = df_maraton24['10 km Czas'].apply(convert_time_to_seconds)
df_maraton24['15 km Czas'] = df_maraton24['15 km Czas'].apply(convert_time_to_seconds) 
df_maraton24['20 km Czas'] = df_maraton24['20 km Czas'].apply(convert_time_to_seconds) 

In [14]:
df_maraton24.sample(10)

,Miejsce,Numer startowy,Imię,Nazwisko,Miasto,Kraj,Drużyna,Płeć,Płeć Miejsce,Kategoria wiekowa,...,10 km Tempo,15 km Czas,15 km Miejsce Open,15 km Tempo,20 km Czas,20 km Miejsce Open,20 km Tempo,Tempo Stabilność,Czas,Tempo
1521,1522,2142,BARTŁOMIEJ,ŚWIĘCKI,WROCŁAW,POL,TYMUŚKI,M,1407,M40,...,4.620000,4270,1319.0,4.840000,5850,1427.0,5.266667,0.034000,6238,4.928498
2730,2731,7754,OSKAR,GONTA,WROCŁAW,POL,AAT,M,2448,M30,...,4.986667,4615,2668.0,5.270000,6269,2691.0,5.513333,0.028867,6654,5.257170
765,766,274,ŁUKASZ,ZACHARA,KILIANÓW,POL,KLUB BIEGACZA KĄTY WROCŁAWSKIE,M,727,M40,...,4.300000,3964,578.0,4.646667,5443,674.0,4.930000,0.046733,5838,4.612467
1265,1266,1748,ARTUR,SADŁOWSKI,SKOMIELNA BIAŁA,POL,TEAM KOJOT,M,1182,M20,...,4.576667,4186,1063.0,4.886667,5745,1231.0,5.196667,0.048600,6100,4.819467
5767,5768,4221,Anonimowy,ZAWODNIK,NaN,POL,ŚWIŃSKI BIEG,M,4746,M20,...,5.746667,5257,5997.0,6.283333,7129,5889.0,6.240000,0.055533,7521,5.942166
5458,5459,8286,ZUZANNA,GADKOWSKA,WSCHOWA,POL,NaN,K,927,K20,...,5.620000,5180,5645.0,5.916667,6996,5469.0,6.053333,0.025333,7412,5.856048
616,617,666,PIOTR,CHOLEWSKI,WYSZKÓW,POL,KB WYSZKÓW,M,590,M30,...,4.286667,3948,552.0,4.486667,5377,595.0,4.763333,0.026600,5705,4.507387
7380,7381,8305,AGNIESZKA,ZIENKIEWICZ,OSINY,POL,WIECZRNE BIEGANIE W OPOLU,K,1626,K40,...,6.166667,5565,7361.0,6.350000,7601,7388.0,6.786667,0.048867,8061,6.368808
11557,0,26572,JACEK,LIPCZYŃSKI,NaN,NaN,NaN,M,0,M40,...,NaN,0,NaN,NaN,0,NaN,NaN,NaN,0,NaN
8543,8544,3111,KACPER,MEDYGRAL,WROCŁAW,POL,#PLANBIEGANIE,M,6352,M30,...,6.093333,5607,7524.0,6.860000,8035,8398.0,8.093333,0.156733,8598,6.793079


# Wypełniamy brakujące wartości w danych demograficznych

Braki w danych demograficznych zostały uzupełnione wartościami typowymi dla tego zbioru (dominanta). Dzięki temu model klastrowania może przetworzyć wszystkie rekordy, zachowując spójność z profilem typowego uczestnika maratonu

In [15]:
df_maraton23['Płeć'] = df_maraton23['Płeć'].fillna(bezpieczna_dominanta(df_maraton23['Płeć'], 'K'))
df_maraton23['Kategoria wiekowa'] = df_maraton23['Kategoria wiekowa'].fillna(bezpieczna_dominanta(df_maraton23['Kategoria wiekowa'], 'M0')) 
df_maraton23['Rocznik'] = df_maraton23['Rocznik'].fillna(bezpieczna_dominanta(df_maraton23['Rocznik'], '1970'))

In [16]:
df_maraton24['Płeć'] = df_maraton24['Płeć'].fillna(bezpieczna_dominanta(df_maraton24['Płeć'], 'K'))
df_maraton24['Kategoria wiekowa'] = df_maraton24['Kategoria wiekowa'].fillna(bezpieczna_dominanta(df_maraton24['Kategoria wiekowa'], 'M0')) 
df_maraton24['Rocznik'] = df_maraton24['Rocznik'].fillna(bezpieczna_dominanta(df_maraton24['Rocznik'], '1970'))

# Wypełnianie brakujących wartości w statystykach biegowych

W celu uzupełnienia braków w międzyczasach zastosowano interpolację liniową. Wybór tej metody podyktowany jest charakterystyką danych procesowych (bieg ciągły) – pozwala ona na oszacowanie brakujących wartości w oparciu o dynamikę tempa konkretnego zawodnika pomiędzy najbliższymi znanymi punktami pomiarowymi, co minimalizuje ryzyko wprowadzenia błędów do analizy stabilności biegu.

In [17]:
# Lista kolumn do interpolacji
kolumny_czasowe = ['Czas', '5 km Czas', '10 km Czas', '15 km Czas', '20 km Czas','5 km Tempo','10 km Tempo','15 km Tempo','20 km Tempo','Tempo','Tempo Stabilność']

# 1. Przygotowanie danych i zamiana zer na NaN
temp_df = df_maraton23.copy()
for col in kolumny_czasowe:
    temp_df[col] = pd.to_numeric(temp_df[col], errors='coerce').replace(0, np.nan)

# 2. Sortowanie
temp_df = temp_df.sort_values(by=['Płeć', 'Kategoria wiekowa', 'Numer startowy'])

# 3. Interpolacja CAŁEGO DataFrame, a następnie wypełnianie grupowe (bezpieczna kolejność)
# Interpolacja globalna (ominięcie problemu grup)
temp_df[kolumny_czasowe] = temp_df[kolumny_czasowe].interpolate(method='linear', limit_direction='both')

# 4. Wypełnianie pozostałych braków średnią grupową
# Użycie transform jest bezpieczne, bo fillna nie generuje błędów metody
temp_df[kolumny_czasowe] = temp_df.groupby(['Płeć', 'Kategoria wiekowa'])[kolumny_czasowe].transform(
    lambda x: x.fillna(x.mean())
)

# 5. Finalne wypełnienie
df_maraton23 = temp_df
df_maraton23 = df_maraton23.fillna('Brak') 
df_maraton23.sample(10)

,Miejsce,Numer startowy,Imię,Nazwisko,Miasto,Kraj,Drużyna,Płeć,Płeć Miejsce,Kategoria wiekowa,...,10 km Tempo,15 km Czas,15 km Miejsce Open,15 km Tempo,20 km Czas,20 km Miejsce Open,20 km Tempo,Tempo Stabilność,Czas,Tempo
2329,2330,9271,RADOSŁAW,ADAMSKI,ŁÓDŹ,POL,Brak,M,2095,M40,...,5.040000,4617.0,2477.0,5.283333,6310.0,2370.0,5.643333,0.039467,6603.0,5.216876
2173,2174,9798,MARCIN,DĄBROWSKI,WROCŁAW,POL,MATNER RUNNING TEAM,M,1963,M40,...,4.853333,4485.0,1992.0,5.286667,6230.0,2130.0,5.816667,0.069067,6549.0,5.174212
5532,5533,2163,JAKUB,KŁOSSOWSKI,WARSZAWA,POL,Brak,M,4419,M30,...,6.100000,5506.0,6015.0,6.170000,7457.0,5635.0,6.503333,0.026600,7784.0,6.149957
792,793,6380,ALEKSANDR,RASCHINSKIY,WARSZAWA,BLR,MIKKELLER RUNNING CLUB,M,739,M30,...,4.513333,4147.0,897.0,4.610000,5638.0,803.0,4.970000,0.018133,5903.0,4.663822
8251,0,3470,MARCIN,DETNERSKI,Brak,Brak,Brak,M,0,M40,...,5.178333,4814.5,Brak,5.528333,6669.5,Brak,6.183333,0.057500,7010.0,5.538437
5565,5566,5948,IRENA,KUNICKA,BYSTRZYCA,POL,Brak,K,1124,K50,...,6.040000,5339.0,5453.0,6.230000,7398.0,5485.0,6.863333,0.084000,7798.0,6.161018
6901,6902,5172,GRZEGORZ,RODER,LEGNICA,POL,Brak,M,5193,M50,...,6.796667,6034.0,7166.0,6.750000,8291.0,6984.0,7.523333,0.056467,8649.0,6.833373
1703,1704,918,MARCIN,PIESZAK,WROCŁAW,POL,CAMPUS DOMASŁAWICE TEAM,M,1560,M40,...,4.656667,4304.0,1315.0,5.113333,6007.0,1548.0,5.676667,0.075133,6363.0,5.027258
6332,6333,7382,MARCIN,BLIMEL,DĄBRÓWKA,POL,Brak,M,4896,M40,...,5.610000,5398.0,5647.0,6.810000,7777.0,6264.0,7.930000,0.165400,8218.0,6.492850
6227,6228,4200,ANDRZEJ,KOWALSKI,WARSZAWA,POL,NAPRZÓD MŁOCINY,M,4830,M40,...,6.200000,5658.0,6390.0,6.550000,7777.0,6258.0,7.063333,0.064200,8141.0,6.432014


In [18]:
# Lista kolumn do interpolacji
kolumny_czasowe = ['Czas', '5 km Czas', '10 km Czas', '15 km Czas', '20 km Czas','5 km Tempo','10 km Tempo','15 km Tempo','20 km Tempo','Tempo','Tempo Stabilność']

# 1. Przygotowanie danych i zamiana zer na NaN
temp_df = df_maraton24.copy()
for col in kolumny_czasowe:
    temp_df[col] = pd.to_numeric(temp_df[col], errors='coerce').replace(0, np.nan)

# 2. Sortowanie
temp_df = temp_df.sort_values(by=['Płeć', 'Kategoria wiekowa', 'Numer startowy'])

# 3. Interpolacja CAŁEGO DataFrame, a następnie wypełnianie grupowe (bezpieczna kolejność)
# Interpolacja globalna (ominięcie problemu grup)
temp_df[kolumny_czasowe] = temp_df[kolumny_czasowe].interpolate(method='linear', limit_direction='both')

# 4. Wypełnianie pozostałych braków średnią grupową
# Użycie transform jest bezpieczne, bo fillna nie generuje błędów metody
temp_df[kolumny_czasowe] = temp_df.groupby(['Płeć', 'Kategoria wiekowa'])[kolumny_czasowe].transform(
    lambda x: x.fillna(x.mean())
)


# 5. Finalne wypełnienie
df_maraton24 = temp_df
df_maraton24 = df_maraton24.fillna('Brak') 
df_maraton24.sample(10)

,Miejsce,Numer startowy,Imię,Nazwisko,Miasto,Kraj,Drużyna,Płeć,Płeć Miejsce,Kategoria wiekowa,...,10 km Tempo,15 km Czas,15 km Miejsce Open,15 km Tempo,20 km Czas,20 km Miejsce Open,20 km Tempo,Tempo Stabilność,Czas,Tempo
10348,0,4194,MICHAŁ,BALCERZAK,Brak,Brak,Balcerzak Running Team,M,0,M30,...,6.513333,5937.000000,Brak,7.001667,8269.500000,Brak,7.775000,0.099767,8754.500000,6.916726
6947,6948,8529,DAMIAN,LOBA,POZNAŃ,POL,ŚWIERCZEWO RUN,M,5506,M50,...,5.690000,5349.000000,6429.0,6.360000,7405.000000,6808.0,6.853333,0.077800,7894.000000,6.236865
1297,1298,5331,LESŁAW,LEŚNIAK,KIELCE ŚWIĘTOKRZYSKIE,POL,ZABIEGANI W KIELCACH,M,1210,M40,...,4.653333,4303.000000,1426.0,4.713333,5776.000000,1295.0,4.910000,-0.002800,6118.000000,4.833689
11233,0,8337,HYELYUN,KIM,Brak,Brak,Lg Energy Solution Wroław,K,0,K40,...,5.777778,5373.000000,Brak,6.035556,7203.333333,Brak,6.101111,0.005422,7607.333333,6.010376
12224,0,23790,DARIUSZ,RZEPA,Brak,Brak,Brak,M,0,M40,...,5.475102,5005.431925,Brak,5.435908,6749.427230,Brak,5.813318,0.001589,7150.082160,5.649113
12982,0,21521,DANIEL,ŁĄGIEWKA,Brak,Brak,Press Glass Sport And Support,M,0,M40,...,5.670438,5166.399061,Brak,5.528013,6937.840376,Brak,5.904804,-0.009933,7341.276995,5.800171
12173,0,10289,KRZYSZTOF,ROJKIEWICZ,Brak,Brak,Brak,M,0,M50,...,6.236667,5756.500000,Brak,6.603333,7805.500000,Brak,6.830000,0.036233,8259.000000,6.525243
6843,6844,9840,MAŁGORZATA,SŁOMIŃSKA,STARGARD SZCZECIŃSKI,POL,BIEGUNKA,K,1404,K20,...,6.010000,5525.000000,7214.0,6.213333,7429.000000,6884.0,6.346667,0.013267,7861.000000,6.210792
298,299,756,Anonimowy,ZAWODNIK,Brak,POL,1985,M,283,M30,...,4.106667,3779.000000,294.0,4.260000,5108.000000,294.0,4.430000,0.015067,5415.000000,4.278265
9001,9002,4462,ANNA,KORCZYŃSKA,LEGNICA,POL,BBL LEGNICA,K,2397,K30,...,6.480000,5989.000000,8702.0,7.240000,8324.000000,8905.0,7.783333,0.107600,8893.000000,7.026152


# Ranking i obsługa miejsc

Dla kolumn określających pozycję zawodników (Open, Płeć oraz Międzyczasy) zastosowano metodę rankingu 'min' (Competition Ranking). Wybór ten podyktowany jest chęcią zachowania spójności z oficjalnymi zasadami sędziowania zawodów sportowych, gdzie zawodnicy z identycznym czasem zajmują tę samą pozycję, a kolejny numer miejsca jest odpowiednio przesunięty.
W procesie modelowania cechy te zostały wyłączone z obliczeń (ignored features), aby uniknąć redundancji danych z kolumnami czasowymi, jednak zachowano je jako kluczowy kontekst dla modelu językowego (LLM) przy generowaniu opisów klastrów.

In [19]:
df_maraton23['Miejsce'] = df_maraton23['Czas'].rank(method='min').astype('Int64')
df_maraton23['Płeć Miejsce'] = df_maraton23.groupby('Płeć')['Czas'].rank(method='min').astype('Int64')
df_maraton23['5 km Miejsce Open'] = df_maraton23['5 km Czas'].rank(method='min').astype('Int64')
df_maraton23['10 km Miejsce Open'] = df_maraton23['10 km Czas'].rank(method='min').astype('Int64')
df_maraton23['15 km Miejsce Open'] = df_maraton23['15 km Czas'].rank(method='min').astype('Int64')
df_maraton23['20 km Miejsce Open'] = df_maraton23['20 km Czas'].rank(method='min').astype('Int64')

df_maraton24['Miejsce'] = df_maraton24['Czas'].rank(method='min').astype('Int64')
df_maraton24['Płeć Miejsce'] = df_maraton24.groupby('Płeć')['Czas'].rank(method='min').astype('Int64')
df_maraton24['5 km Miejsce Open'] = df_maraton24['5 km Czas'].rank(method='min').astype('Int64')
df_maraton24['10 km Miejsce Open'] = df_maraton24['10 km Czas'].rank(method='min').astype('Int64')
df_maraton24['15 km Miejsce Open'] = df_maraton24['15 km Czas'].rank(method='min').astype('Int64')
df_maraton24['20 km Miejsce Open'] = df_maraton24['20 km Czas'].rank(method='min').astype('Int64')


In [20]:
#kolumny = ['Imię', 'Nazwisko', 'Numer startowy', 'Kategoria wiekowa',"Czas"]
#df_maraton23.loc[df_maraton23['Kategoria wiekowa'].isna(), kolumny]
df_maraton24.sample(10)

,Miejsce,Numer startowy,Imię,Nazwisko,Miasto,Kraj,Drużyna,Płeć,Płeć Miejsce,Kategoria wiekowa,...,10 km Tempo,15 km Czas,15 km Miejsce Open,15 km Tempo,20 km Czas,20 km Miejsce Open,20 km Tempo,Tempo Stabilność,Czas,Tempo
5803,6830,4310,TOMASZ,GULBIERZ,WROCŁAW,POL,Brak,M,5624,M40,...,5.640000,5275.00,7233,6.000000,7133.00,6961,6.193333,0.022200,7530.0,5.949277
1169,1225,3901,PAWEŁ,MILEWSKI,WROCŁAW,POL,Brak,M,1152,M30,...,4.546667,4257.00,1363,4.856667,5732.00,1262,4.916667,0.014000,6051.0,4.780754
6456,7681,5114,MIROSŁAW,HOŁDYŃSKI,KĄTY WROCŁAWSKIE,POL,WKB META,M,6149,M60,...,5.583333,5257.00,7129,6.270000,7268.00,7576,6.703333,0.075733,7731.0,6.108082
6588,7884,8159,ZBIGNIEW,HOFFMANN,LESZNO,POL,Brak,M,6284,M40,...,5.863333,5411.00,8119,6.053333,7351.00,7955,6.466667,0.024600,7779.0,6.146006
8186,10326,8398,KAROL,ORZECHOWSKI,KOBYLICE,POL,AKS BUKOWE,M,7727,M30,...,6.400000,5832.00,10334,6.833333,7952.00,10357,7.066667,0.060267,8414.0,6.647705
10808,11021,23268,EWA,FUCHS,Brak,Brak,Brak,K,2965,K20,...,6.389067,5978.64,10952,7.082800,8197.28,11000,7.395467,0.070187,8709.0,6.880777
7409,9115,9951,MICHAŁ,ZIEGLER,WYSOKA,POL,Brak,M,7046,M30,...,5.986667,5564.00,9001,6.346667,7608.00,9071,6.813333,0.043200,8075.0,6.379869
902,944,2564,MARCIN,BŁASZCZAK,PSZCZEW,POL,Brak,M,895,M30,...,4.600000,4177.00,1072,4.623333,5602.00,958,4.750000,0.003467,5929.0,4.684364
6191,7297,4296,MARCIN,JURCZENKO,KRAKÓW,POL,ITMBW KRAKÓW,M,5902,M40,...,6.143333,5539.00,8862,5.846667,7267.00,7571,5.760000,-0.048733,7648.0,6.042506
5244,6106,10852,ŁUKASZ,JABŁOŃSKI,WROCŁAW,POL,CHŁOSTA,M,5153,M30,...,5.616667,5196.00,6761,6.033333,6951.00,6223,5.850000,0.019133,7341.0,5.799953


# Walidacja Schematu Danych (Pandera)

Po zakończeniu procesu preprocessingu i inżynierii cech, dane zostają poddane rygorystycznej walidacji schematu przy użyciu biblioteki Pandera.
Cel tego kroku:

    Gwarancja spójności typów: Upewnienie się, że kolumny numeryczne (np. Tempo, Czas) są faktycznie liczbami, a kategoryczne (np. Płeć) zawierają tylko dozwolone wartości ("M", "K").
    Zabezpieczenie modelu ML: Model klastrujący oczekuje konkretnego zestawu wejściowego. Walidacja schematu zapobiega błędom w fazie predykcji, które mogłyby wyniknąć ze zmiany formatu danych wejściowych.
    Obsługa braków: Jawne zdefiniowanie parametrów nullable=True pozwala na kontrolowany nadzór nad rekordami, które posiadają niepełne informacje.

Jest to ostatnia barierka ochronna przed wdrożeniem aplikacji na produkcję w DigitalOcean, gwarantująca, że skrypt nie przerwie pracy z powodu niespodziewanej zmiany w strukturze plików CSV.

In [21]:
schema = pa.DataFrameSchema(
    columns={
        "Miejsce": pa.Column(pa.Int, nullable=True),
        "Numer startowy": pa.Column(pa.Int, nullable=True),
        "Imię": pa.Column(pa.String),
        "Nazwisko": pa.Column(pa.String),
        "Miasto": pa.Column(pa.String, nullable=True),
        "Kraj": pa.Column(pa.String, nullable=True),
        "Drużyna": pa.Column(pa.String, nullable=True),
        "Płeć": pa.Column(pa.String, pa.Check.isin(["M", "K"])),
        "Płeć Miejsce": pa.Column(pa.Int, nullable=True),
        "Kategoria wiekowa": pa.Column(pa.String),
        "Kategoria wiekowa Miejsce": pa.Column(pa.Int, nullable=True),
        "Rocznik": pa.Column(pa.Int, nullable=True),
        "5 km Czas": pa.Column(pa.Float),
        "5 km Miejsce Open": pa.Column(pa.Int, nullable=True),
        "5 km Tempo": pa.Column(pa.Float),
        "10 km Czas": pa.Column(pa.Float),
        "10 km Miejsce Open": pa.Column(pa.Int, nullable=True),
        "10 km Tempo": pa.Column(pa.Float),
        "15 km Czas": pa.Column(pa.Float),
        "15 km Miejsce Open": pa.Column(pa.Int, nullable=True),
        "15 km Tempo": pa.Column(pa.Float),
        "20 km Czas": pa.Column(pa.Float),
        "20 km Miejsce Open": pa.Column(pa.Int, nullable=True),
        "20 km Tempo": pa.Column(pa.Float),
        "Tempo Stabilność": pa.Column(pa.Float, nullable=True),
        "Czas": pa.Column(pa.Float),
        "Tempo": pa.Column(pa.Float, nullable=True),
    },
    strict=True, # Sprawdza, czy nie ma nadmiarowych kolumn
    coerce=True  # Automatycznie próbuje naprawić typy (np. str na int)
)

In [22]:
try:
    schema.validate(df_maraton23)
    print("Dane maraton23 są poprawne")
except pa.errors.SchemaError as e:
    print(f"Błąd walidacji: {e}")

Dane maraton23 są poprawne


In [23]:
try:
    schema.validate(df_maraton24)
    print("Dane maraton24 są poprawne")
except pa.errors.SchemaError as e:
    print(f"Błąd walidacji: {e}")


Dane maraton24 są poprawne


# Inżynieria Cech i Preprocessing Końcowy

Funkcja preprocessing_maraton odpowiada za ostateczne przygotowanie danych przed walidacją schematu i modelowaniem. Kluczowe operacje to:

    Normalizacja typów danych: Konwersja kolumn kategorycznych na typ tekstowy oraz rygorystyczne wymuszenie typów numerycznych dla parametrów czasowych (eliminacja błędów formatowania CSV).
    Wyliczanie cech pochodnych (Feature Engineering):
        Tempo: Obliczenie średniego tempa biegu na podstawie czasu całkowitego i dystansu maratońskiego (42.195 km).
        Tempo_Stabilnosc: Wyznaczenie odchylenia standardowego z międzyczasów na poszczególnych odcinkach. Jest to kluczowa cecha określająca strategię biegu zawodnika (tzw. pacing), pozwalająca modelowi ML na odróżnienie biegaczy stabilnych od tych, którzy drastycznie zwalniają w drugiej połowie dystansu.

In [24]:
def preprocessing_maraton(df: pd.DataFrame) -> pd.DataFrame:
    # 1. Konwersja kolumn tekstowych (z użyciem pętli dla czystości)
    cols_to_str = ['Imię', 'Nazwisko', 'Miasto', 'Drużyna', 'Kraj']
    for col in cols_to_str:
        if col in df.columns:
            df[col] = df[col].astype('string')
    
    # 2. Bezpieczne obliczanie średniego tempa
    if 'Czas' in df.columns:
        # Wymuszamy typ numeryczny (coerce zamieni błędy na NaN)
        df['Czas'] = pd.to_numeric(df['Czas'], errors='coerce')
        df['Tempo'] = df['Czas'] / 42.195

    # 3. Obliczanie Stabilności na dostępnych odcinkach
    odcinki_tempo = ['5 km Tempo', '10 km Tempo', '15 km Tempo', '20 km Tempo']
    # Bierzemy tylko te kolumny, które faktycznie są w DataFrame
    existing_split_cols = [c for c in odcinki_tempo if c in df.columns]
    
    if existing_split_cols:
        # Wymuszamy typ numeryczny dla międzyczasów
        for col in existing_split_cols:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        # Obliczamy odchylenie standardowe (miara stabilności)
        df['Tempo Stabilność'] = df[existing_split_cols].std(axis=1)

    return df

In [25]:
preprocessing_maraton(df_maraton23)
preprocessing_maraton(df_maraton24)


,Miejsce,Numer startowy,Imię,Nazwisko,Miasto,Kraj,Drużyna,Płeć,Płeć Miejsce,Kategoria wiekowa,...,10 km Tempo,15 km Czas,15 km Miejsce Open,15 km Tempo,20 km Czas,20 km Miejsce Open,20 km Tempo,Tempo Stabilność,Czas,Tempo
1189,1248,19,KAROLINA,STAWARZ,OLEŚNICA,POL,KARUN,K,78,K20,...,4.690000,4263.000000,1377,4.860000,5744.000000,1288,4.936667,0.133250,6060.000000,143.618912
378,397,225,MILENA,BATOR,ZAWONIA,POL,OŚ RACING TEAM,K,20,K20,...,4.170000,3854.000000,426,4.380000,5208.000000,409,4.513333,0.144248,5501.000000,130.370897
215,227,380,OLHA,PUHACHOVA,KYIV,UKR,TS REGLE SZKLARSKA POREBA,K,7,K20,...,4.033333,3725.000000,228,4.216667,5025.000000,223,4.333333,0.124257,5321.000000,126.104989
20,23,387,NATALIJA,SEMENOVYCH,KIJÓW,UKR,Brak,K,1,K20,...,3.516667,3227.000000,22,3.716667,4362.000000,23,3.783333,0.135578,4616.000000,109.396848
397,416,403,AGATA,MAJ,Brak,NOR,Brak,K,22,K20,...,4.183333,3870.000000,447,4.376667,5226.000000,423,4.520000,0.138310,5525.000000,130.939685
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10642,11540,85668,JANUSZ,DALEK,Brak,Brak,Brak,M,8326,M70,...,6.794848,6228.636364,11642,7.490303,8487.727273,11564,7.530303,0.521500,8977.818182,212.769716
10638,11508,86230,JERZY,CZYNIEWSKI,Brak,Brak,Brak,M,8307,M70,...,6.792424,6226.318182,11639,7.491818,8476.863636,11549,7.501818,0.516760,8963.909091,212.440078
9096,11482,9148,STANISŁAW,BEDNARSKI,STRZELIN,POL,Brak,M,8295,M80,...,6.790000,6224.000000,11630,7.493333,8466.000000,11525,7.473333,0.512326,8950.000000,212.110440
11974,11482,10897,MARIAN,PAWLACZYK,Brak,Brak,Wkb Piast Wrocław,M,8295,M80,...,6.790000,6224.000000,11630,7.493333,8466.000000,11525,7.473333,0.512326,8950.000000,212.110440


# Egzekucja Walidacji i Raportowanie Błędów

Poniższy blok kodu uruchamia proces weryfikacji danych dla obu roczników maratonu. Wykorzystanie trybu Lazy Validation pozwala na przechwycenie wszystkich niezgodności ze schematem jednocześnie. W przypadku wykrycia błędów (np. niewłaściwych typów danych lub naruszenia reguł biznesowych), generowany jest szczegółowy raport failure_cases, który wskazuje precyzyjną lokalizację problematycznych rekordów.

In [41]:
try:
    schema.validate(df_maraton23, lazy=True)
    schema.validate(df_maraton24, lazy=True)
except pa.errors.SchemaErrors as err:
    print("Walidacja nie powiodła się. Znalezione błędy:")
    # Pokazuje, które wiersze i kolumny nie przeszły walidacji
    print(err.failure_cases) 
    # Pokazuje oryginalne dane, które wywołały błąd
    #print(err.data)
    #print(err.data.head()) 

Walidacja nie powiodła się. Znalezione błędy:
  column  failure_case index   schema_context             check check_number
0   None  Rok_maratonu  None  DataFrameSchema  column_in_schema         None


# Anonimizacja Danych Wrażliwych (Faker)

W celu ochrony prywatności uczestników maratonu oraz spełnienia standardów bezpieczeństwa danych (zgodność z RODO), zastosowano proces anonimizacji.
Dlaczego Faker?

    Eliminacja danych wrażliwych: Prawdziwe imiona i nazwiska biegaczy zostały zastąpione danymi syntetycznymi wygenerowanymi przez bibliotekę Faker.
    Zachowanie realizmu: W przeciwieństwie do zwykłego usuwania kolumn, Faker pozwala zachować strukturę danych (np. rozróżnienie płci na podstawie imion), co jest istotne dla modelu językowego (LLM) podczas generowania opisów klastrów.
    Bezpieczeństwo w chmurze: Dzięki temu procesowi, zbiory danych przechowywane w DigitalOcean Spaces oraz przesyłane do OpenAI API nie zawierają żadnych informacji pozwalających na identyfikację konkretnych osób fizycznych.

In [27]:
fake = Faker('pl_PL') 

def anonymize_marathon_data(df: pd.DataFrame) -> pd.DataFrame:
    # Tworzymy kopię, żeby nie modyfikować oryginalnych danych
    df_anonymized = df.copy()
    
    # 1. Anonimizacja danych osobowych
    if 'Imię' in df.columns:
        df_anonymized['Imię'] = [fake.first_name() for _ in range(len(df))]
        
    if 'Nazwisko' in df.columns:
        df_anonymized['Nazwisko'] = [fake.last_name() for _ in range(len(df))]

    # 2. Anonimizacja danych geograficznych/organizacyjnych
    if 'Miasto' in df.columns:
        df_anonymized['Miasto'] = [fake.city() for _ in range(len(df))]
        
    if 'Drużyna' in df.columns:
        # Można użyć nazw firm jako nazw drużyn
        df_anonymized['Drużyna'] = [fake.company() for _ in range(len(df))]

    # Jeśli zdecydujesz się anonimizować kraj:
    # if 'Kraj' in df.columns:
    #     df_anonymized['Kraj'] = [fake.country_code() for _ in range(len(df))]

    return df_anonymized

In [28]:
df_maraton23 = anonymize_marathon_data(df_maraton23)
df_maraton23

,Miejsce,Numer startowy,Imię,Nazwisko,Miasto,Kraj,Drużyna,Płeć,Płeć Miejsce,Kategoria wiekowa,...,10 km Tempo,15 km Czas,15 km Miejsce Open,15 km Tempo,20 km Czas,20 km Miejsce Open,20 km Tempo,Tempo Stabilność,Czas,Tempo
6551,7207,100,Rozalia,Białasik,Mława,POL,Ciepłuch-Horbacz S.A.,K,1748,K20,...,6.040000,5559.0,6750,6.613333,7936.0,7156,7.923333,0.928707,8376.0,198.506932
1029,1066,117,Jędrzej,Rejniak,Lubliniec,POL,Fundacja Marko,K,73,K20,...,4.606667,4278.0,1306,4.816667,5789.0,1106,5.036667,0.175760,6047.0,143.310819
8138,8939,119,Kazimierz,Wawrzonek,Słupsk,POL,FPUH Zontek,K,2607,K20,...,8.570000,7747.0,8934,9.476667,10995.0,8939,10.826667,1.308911,11581.0,274.463799
6523,7177,121,Aleksander,Kunert,Wyszków,POL,Joszko Sp.k.,K,1732,K20,...,6.210000,5708.0,7154,6.636667,7940.0,7163,7.440000,0.587178,8351.0,197.914445
8215,7292,143,Alex,Pawlonka,Nowy Dwór Mazowiecki,Brak,Grupa Szatko,K,1791,K20,...,6.286667,5770.5,7293,6.751667,8019.0,7272,7.495000,0.593762,8433.5,199.869653
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6448,7092,8535,Natasza,Gołoś,Jawor,POL,Senator i syn s.c.,M,5398,M70,...,6.060000,5559.0,6750,6.443333,7874.0,7057,7.716667,0.792869,8293.0,196.539874
7229,7971,8703,Emil,Wyskiel,Kościerzyna,POL,Spółdzielnia Maszota,M,5855,M70,...,7.033333,6168.0,8118,7.133333,8502.0,7971,7.780000,0.567578,8940.0,211.873445
2188,2314,8965,Fabian,Kwapisiewicz,Warszawa,POL,Gabinety Boguś,M,2092,M70,...,4.856667,4407.0,1761,5.226667,6211.0,2194,6.013333,0.613680,6556.0,155.373859
8136,8937,2797,Maksymilian,Wacławiak,Turek,POL,Gabinety Trzyna Sp.k.,M,6331,M80,...,8.766667,7773.0,8936,9.013333,10810.0,8937,10.123333,0.831289,11370.0,269.463207


In [29]:
df_maraton24 = anonymize_marathon_data(df_maraton24)
df_maraton24

,Miejsce,Numer startowy,Imię,Nazwisko,Miasto,Kraj,Drużyna,Płeć,Płeć Miejsce,Kategoria wiekowa,...,10 km Tempo,15 km Czas,15 km Miejsce Open,15 km Tempo,20 km Czas,20 km Miejsce Open,20 km Tempo,Tempo Stabilność,Czas,Tempo
1189,1248,19,Sandra,Więcławek,Grudziądz,POL,Stowarzyszenie Grzelec,K,78,K20,...,4.690000,4263.000000,1377,4.860000,5744.000000,1288,4.936667,0.133250,6060.000000,143.618912
378,397,225,Anita,Frycz,Żagań,POL,Kluczek Sp. z o.o. Sp.k.,K,20,K20,...,4.170000,3854.000000,426,4.380000,5208.000000,409,4.513333,0.144248,5501.000000,130.370897
215,227,380,Malwina,Starościak,Kłodzko,UKR,PPUH Romaniak Sp.k.,K,7,K20,...,4.033333,3725.000000,228,4.216667,5025.000000,223,4.333333,0.124257,5321.000000,126.104989
20,23,387,Urszula,Nyc,Wałcz,UKR,FPUH Niedośpiał-Kawala S.A.,K,1,K20,...,3.516667,3227.000000,22,3.716667,4362.000000,23,3.783333,0.135578,4616.000000,109.396848
397,416,403,Maurycy,Doktor,Gliwice,NOR,Grupa Majdak i syn s.c.,K,22,K20,...,4.183333,3870.000000,447,4.376667,5226.000000,423,4.520000,0.138310,5525.000000,130.939685
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10642,11540,85668,Przemysław,Kopik,Gorzów Wielkopolski,Brak,PPUH Rolek Sp.j.,M,8326,M70,...,6.794848,6228.636364,11642,7.490303,8487.727273,11564,7.530303,0.521500,8977.818182,212.769716
10638,11508,86230,Marek,Noras,Chełm,Brak,Spółdzielnia Piwowarek i syn s.c.,M,8307,M70,...,6.792424,6226.318182,11639,7.491818,8476.863636,11549,7.501818,0.516760,8963.909091,212.440078
9096,11482,9148,Stanisław,Filus,Jawor,POL,FPUH Prokopiak Sp.j.,M,8295,M80,...,6.790000,6224.000000,11630,7.493333,8466.000000,11525,7.473333,0.512326,8950.000000,212.110440
11974,11482,10897,Piotr,Wojewodzic,Myszków,Brak,Fundacja Kuba-Martynowicz S.A.,M,8295,M80,...,6.790000,6224.000000,11630,7.493333,8466.000000,11525,7.473333,0.512326,8950.000000,212.110440


In [30]:
df_maraton23['Rok_maratonu'] = 2023
df_maraton24['Rok_maratonu'] = 2024
df_maraton = pd.concat([df_maraton23, df_maraton24], ignore_index=True)
df_maraton

,Miejsce,Numer startowy,Imię,Nazwisko,Miasto,Kraj,Drużyna,Płeć,Płeć Miejsce,Kategoria wiekowa,...,15 km Czas,15 km Miejsce Open,15 km Tempo,20 km Czas,20 km Miejsce Open,20 km Tempo,Tempo Stabilność,Czas,Tempo,Rok_maratonu
0,7207,100,Rozalia,Białasik,Mława,POL,Ciepłuch-Horbacz S.A.,K,1748,K20,...,5559.000000,6750,6.613333,7936.000000,7156,7.923333,0.928707,8376.000000,198.506932,2023
1,1066,117,Jędrzej,Rejniak,Lubliniec,POL,Fundacja Marko,K,73,K20,...,4278.000000,1306,4.816667,5789.000000,1106,5.036667,0.175760,6047.000000,143.310819,2023
2,8939,119,Kazimierz,Wawrzonek,Słupsk,POL,FPUH Zontek,K,2607,K20,...,7747.000000,8934,9.476667,10995.000000,8939,10.826667,1.308911,11581.000000,274.463799,2023
3,7177,121,Aleksander,Kunert,Wyszków,POL,Joszko Sp.k.,K,1732,K20,...,5708.000000,7154,6.636667,7940.000000,7163,7.440000,0.587178,8351.000000,197.914445,2023
4,7292,143,Alex,Pawlonka,Nowy Dwór Mazowiecki,Brak,Grupa Szatko,K,1791,K20,...,5770.500000,7293,6.751667,8019.000000,7272,7.495000,0.593762,8433.500000,199.869653,2023
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21952,11540,85668,Przemysław,Kopik,Gorzów Wielkopolski,Brak,PPUH Rolek Sp.j.,M,8326,M70,...,6228.636364,11642,7.490303,8487.727273,11564,7.530303,0.521500,8977.818182,212.769716,2024
21953,11508,86230,Marek,Noras,Chełm,Brak,Spółdzielnia Piwowarek i syn s.c.,M,8307,M70,...,6226.318182,11639,7.491818,8476.863636,11549,7.501818,0.516760,8963.909091,212.440078,2024
21954,11482,9148,Stanisław,Filus,Jawor,POL,FPUH Prokopiak Sp.j.,M,8295,M80,...,6224.000000,11630,7.493333,8466.000000,11525,7.473333,0.512326,8950.000000,212.110440,2024
21955,11482,10897,Piotr,Wojewodzic,Myszków,Brak,Fundacja Kuba-Martynowicz S.A.,M,8295,M80,...,6224.000000,11630,7.493333,8466.000000,11525,7.473333,0.512326,8950.000000,212.110440,2024


# Inżynieria Cech i Przygotowanie Danych (Feature Engineering)

Funkcja prepare_marathon_data pełni kluczową rolę w transformacji surowych danych maratońskich na format wejściowy dla algorytmów uczenia maszynowego. Proces ten obejmuje trzy kluczowe etapy:

    Ekstrakcja Cech Demograficznych (Parsing):
        Przetworzenie kolumny „Kategoria wiekowa” w celu wyodrębnienia numerycznej grupy wiekowej oraz zamiana płci na format One-Hot Encoding (Plec_K, Plec_M). Pozwala to modelowi na matematyczną interpretację różnic biologicznych i wiekowych między biegaczami.
    Selekcja Cech (Feature Selection):
        Wybór unikalnego zestawu zmiennych łączących dynamikę biegu (międzyczasy i czas końcowy) z danymi demograficznymi. Dzięki temu klastrowanie odbywa się w wielowymiarowej przestrzeni uwzględniającej zarówno tempo, jak i profil biegacza.
    Standaryzacja (StandardScaler):
        Kluczowy krok dla algorytmów opartych na odległościach (np. K-Means). Standaryzacja sprowadza cechy o różnych jednostkach (sekundy vs wiek) do wspólnej skali o średniej 0 i odchyleniu 1, co zapobiega dominacji modelu przez zmienne o większych wartościach liczbowych.

In [42]:
def prepare_marathon_data(df):
    # 1. Wyodrębnienie płci i wieku z kolumny 'Kategoria wiekowa' (np. M30 -> M, 30)
    # Korzystamy z [Series.str.extract](https://pandas.pydata.org)
    df['Plec_K'] = df['Płeć'].apply(lambda x: 1 if 'K' in str(x).upper() else 0)
    df['Plec_M'] = df['Płeć'].apply(lambda x: 1 if 'M' in str(x).upper() else 0)
    
    # Wyciągamy same cyfry dla grupy wiekowej
    df['Grupa_Wiekowa'] = df['Kategoria wiekowa'].str.extract('(\d+)').fillna(0).astype(int)

    # 2. Wybór kolumn do uczenia (X)
    # Tu muszą być wszystkie zmienne: czas + wiek + płeć
    features = ['5 km Czas','10 km Czas','15 km Czas','20 km Czas','Czas' ,'Grupa_Wiekowa', 'Plec_K', 'Plec_M']
    X = df[features]

    # 3. Skalowanie - kluczowy krok dla algorytmów dystansowych (K-Means, MeanShift)
    # Używamy [StandardScaler](https://scikit-learn.org)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    return X_scaled, df

x,dfx = prepare_marathon_data(df_maraton)

In [32]:
dfx.sample(10)

,Miejsce,Numer startowy,Imię,Nazwisko,Miasto,Kraj,Drużyna,Płeć,Płeć Miejsce,Kategoria wiekowa,...,20 km Czas,20 km Miejsce Open,20 km Tempo,Tempo Stabilność,Czas,Tempo,Rok_maratonu,Plec_K,Plec_M,Grupa_Wiekowa
13302,1368,2704,Aleks,Dulik,Białogard,POL,Fundacja Chaba S.A.,M,1277,M20,...,5788.00,1387,4.866667,0.109036,6122.0,145.088281,2024,0,1,20
19528,6051,7864,Miłosz,Chojna,Kielce,POL,Grupa Roda,M,5111,M40,...,6882.00,5914,6.326667,0.515439,7325.0,173.598768,2024,0,1,40
9530,10146,10700,Norbert,Kościk,Białystok,POL,Dziczek s.c.,K,2512,K20,...,7891.00,10144,6.850000,0.198967,8352.0,197.938144,2024,1,0,20
7335,60,6909,Patryk,Dering,Bielawa,AUT,PPUH Mila Sp.j.,M,57,M40,...,4653.00,63,4.080000,0.142709,4878.0,115.606114,2023,0,1,40
13689,6308,5832,Krzysztof,Uliasz,Koszalin,POL,Stowarzyszenie Chomiuk-Miszta Sp. z o.o. Sp.k.,M,5294,M20,...,6970.00,6304,5.926667,0.124201,7384.0,174.997038,2024,0,1,20
9806,3407,85332,Daniel,Powała,Sochaczew,Brak,Fundacja Sieczko,K,363,K20,...,6320.96,3211,5.828267,0.406948,6747.0,159.900462,2024,1,0,20
8913,6901,2566,Agnieszka,Dzienisz,Będzin,POL,Ryczko-Cebulak Sp. z o.o. Sp.k.,M,5306,M70,...,7797.00,6897,7.200000,0.497292,8188.0,194.051428,2023,0,1,70
18777,11722,4389,Klara,Kempny,Sieradz,POL,Stowarzyszenie Pawliczak s.c.,M,8420,M40,...,8550.00,11684,8.640000,1.241822,9093.0,215.499467,2024,0,1,40
15606,1041,3748,Igor,Kozień,Kraków,POL,Grupa Wyderka,M,984,M30,...,5652.00,1070,4.923333,0.181700,5970.0,141.485958,2024,0,1,30
2920,6776,3047,Karina,Wojnicz,Oświęcim,POL,Hallmann-Pawela Sp.j.,M,5224,M20,...,7671.00,6667,7.603333,0.881797,8105.0,192.084370,2023,0,1,20


# Upload to digital ocean

Gotowy zbiór treningowy wysyłamy do serwera digital ocean

In [33]:
def upload_df_to_digital_ocean(df,access_key,access_secret_key, nazwa_space,plik_csv,region = 'fra1',end_url='https://fra1.digitaloceanspaces.com'):
    
    session = boto3.session.Session()
    client = session.client(
        's3',
        region_name=region.strip(),
        endpoint_url=end_url.strip(), # Używamy endpointu bazowego do komunikacji API
        aws_access_key_id=access_key.strip(),
        aws_secret_access_key=access_secret_key.strip()
    )
    # 1. Przygotowanie bufora tekstowego (wirtualny plik w RAM)
    csv_buffer = io.StringIO()
    
    # 2. Zapisanie DataFrame do bufora (z Twoim separatorem i kodowaniem)
    df.to_csv(csv_buffer, sep=';', encoding='utf-8-sig', index=False)
    
    try:
        # 3. Wysłanie zawartości bufora do DigitalOcean
        client.put_object(
            Bucket=nazwa_space,
            Key=plik_csv,
            Body=csv_buffer.getvalue(),
            ContentType='text/csv'
        )
        print(f"Plik {plik_csv} wysłany bezpośrednio do {nazwa_space}")
    except Exception as e:
        print(f"Błąd wysyłki: {e}")

In [34]:
df_maraton.to_csv(data_dir / "halfmarathon_wroclaw_final.csv", sep=';', index=False, encoding='utf-8')
do_access_key, do_secret_key, do_space_input, do_region, do_end_url = handle_digital_ocean_keys()
upload_df_to_digital_ocean(df_maraton,do_access_key, do_secret_key, do_space_input, "halfmarathon_wroclaw_final.csv",do_region, do_end_url)

Plik halfmarathon_wroclaw_final.csv wysłany bezpośrednio do maraton


# Trenowanie i budowa modelu treningowego

Najważniejsza funkcja do budowy modelu treningowego. Można w niej wybrać ilość grup klastrów, na które funkcja będzie starać się podzielić. Można też wybrać, czy zastosować normalizację i transformację. 
1) Z-Score Normalization – aby sprowadzić cechy o różnych jednostkach (wiek, sekundy, punkty stabilności) do wspólnej skali, zapewniając każdemu z nich równy wpływ na ostateczny wynik grupowania.
2) Quantile Transformation – w celu zniwelowania asymetrii rozkładów (szczególnie w czasach i tempie) i zbliżenia ich do rozkładu normalnego, co poprawia stabilność algorytmów klastrujących.

Wybór optymalnego modelu klastrowania został oparty na analizie porównawczej następujących wskaźników:

    1) Precision: Miara wiarygodności przypisania – minimalizuje ryzyko błędnego zakwalifikowania biegacza do niewłaściwej grupy profilowej.
    2) Recall: Miara kompletności – gwarantuje, że model potrafi zidentyfikować większość reprezentantów danej charakterystyki biegowej.
    3) F1-Score: Kluczowy wskaźnik optymalizujący kompromis między precyzją a czułością, zapewniający stabilność modelu.
    4) Min_Pct: Parametr kontrolny dbający o odpowiednią liczebność grup, zapobiegający nadmiernej fragmentacji danych.
    5) Homogeneity & Completeness: Pozwalają ocenić, na ile klastry odzwierciedlają naturalne podziały (np. płeć, wiek), mimo że model nie widział tych etykiet podczas treningu.
    6) Cluster Size (Min/Max/Pct): Parametry te gwarantują użyteczność biznesową modelu – eliminujemy rozwiązania, które tworzą zbyt małe (nieistotne statystycznie) lub zbyt duże (mało precyzyjne) grupy.
    7) n_clusters: Kluczowy parametr kontrolujący ziarnistość podziału społeczności maratońskiej.
    

In [35]:
def build_model(MODEL_ID,DATA,y_true,num_clusters=8, normalize=False, transform=False):
    normalize_method ='zscore'
    transformation_method='quantile'
    dane_numeryczne = DATA.select_dtypes(include=['number'])
    s = setup(data = dane_numeryczne,         
        normalize=normalize, normalize_method=normalize_method,
        transformation=transform, transformation_method=transformation_method,
        session_id=123, html=False, verbose=False,memory=False,low_variance_threshold=0.01,n_jobs = -1)
    
    model = create_model(MODEL_ID, num_clusters, verbose=False)

     # Pobieranie etykiet przypisanych przez model
    labels = assign_model(model)['Cluster'] # PyCaret dodaje kolumnę 'Cluster'
    
    # !!! WAŻNE !!!
    # Konwertujemy etykiety klastrów na format liczbowy (0, 1, 2...), bo sklearn tego wymaga
    labels_numeric = labels.str.replace('Cluster ', '').astype(int)
    
    # 1. Liczymy statystyki liczebności (kod z poprzedniej odpowiedzi)
    counts = pd.Series(labels).value_counts()
    count_min_pct = (counts.min() / len(labels)) * 100 
    
    # 2. Liczymy metryki KLASYFIKACJI (nowość!)
    # Scikit-learn potrzebuje y_true i labels jako numeryczne
    # Musisz najpierw upewnić się, że y_true też jest numeryczne (np. 1, 2, 3...)
    # (Zakładam, że masz już LabelEncoder dla y_true wczesniej w skrypcie)
    
    # Ponieważ te metryki działają na klasyfikacji wieloklasowej,
    # używamy parametru 'average="weighted"' lub "macro"
    
    le = LabelEncoder()
    y_true_numeric = le.fit_transform(y_true)

    # accuracy = accuracy_score(y_true_numeric, labels_numeric) # Accuracy jest mylace w klastrowaniu
    precision = precision_score(y_true_numeric, labels_numeric, average='weighted', zero_division=0)
    recall = recall_score(y_true_numeric, labels_numeric, average='weighted', zero_division=0)
    f1 = f1_score(y_true_numeric, labels_numeric, average='weighted', zero_division=0)
    
    # AUC jest trudne do policzenia dla klastrowania/wieloklasowego, pomijamy je na razie
    
    # 3. Dodajemy nowe kolumny do tabeli metryk
    metrics = pull() # Standardowe metryki PyCaret
    metrics['Min_Pct'] = round(count_min_pct, 2)
    metrics['Precision'] = round(precision, 3)
    metrics['Recall'] = round(recall, 3)
    metrics['F1_Score'] = round(f1, 3)
    # 2. Liczymy statystyki liczebności
    counts = pd.Series(labels).value_counts()
    count_min = counts.min()
    count_max = counts.max()
    count_min_pct = (count_min / len(labels)) * 100 # Procentowo
    metrics = pull()

    # Ręczne liczenie brakujących metryk (tych, co miałeś na 0)
    h_score = homogeneity_score(y_true, labels)
    c_score = completeness_score(y_true, labels)

    # Dodanie naszych metryk do tabeli wyników
    metrics['Homogeneity'] = h_score
    metrics['Completeness'] = c_score
    metrics['Min_Cluster_Size'] = count_min
    metrics['Min_Cluster_Pct'] = round(count_min_pct, 2)
    metrics['Max_Cluster_Size'] = count_max
    metrics['n_clusters'] = len(counts)

    name = f"{MODEL_ID.upper()}_k{num_clusters}_n_{normalize}_t_{transform}"
    metrics.index = [name]
    metrics.insert(0,"model",name)

    return model, metrics


# Porównanie i selekcja modeli klastrujących

W projekcie przetestowano cztery fundamentalnie różne podejścia do segmentacji danych, aby znaleźć najbardziej stabilny podział społeczności maratońskiej:

    K-Means (Podział oparty na centrach):
        Dlaczego: Klasyk i standard w klastrowaniu. Szybki i efektywny, gdy grupy mają kształt zbliżony do kulistego. Idealny do wyznaczenia wyraźnych centrów profili biegaczy (np. „typowy amator”).
    BIRCH (Podejście hierarchiczne i skalowalne):
        Dlaczego: Świetnie radzi sobie z dużymi zbiorami danych (ponad 21 tys. rekordów). Buduje drzewo cech, co pozwala na szybszą analizę bez konieczności trzymania wszystkich danych w pamięci RAM. Jest mniej wrażliwy na szum niż K-Means.
    Affinity Propagation (AP - Przekazywanie podobieństwa):
        Dlaczego: W przeciwieństwie do K-Means, ten model nie wymaga wcześniejszego podania liczby klastrów. Sam wybiera „reprezentantów” (egzemplarze) grup, co pozwala odkryć naturalną strukturę danych, której mogliśmy nie przewidzieć.
    MeanShift (Przesunięcie ku gęstości):
        Dlaczego: Algorytm poszukuje obszarów o największym zagęszczeniu biegaczy w przestrzeni cech. Jest bardzo dobry w ignorowaniu outlierów (np. osób o absurdalnych czasach) i skupianiu się na realnych trendach wewnątrz grup.


In [37]:
results_storage = {}
all_metrics_list = [] # Lista pomocnicza do zbiorczej tabeli

# Przykładowe dane (użyj swojego df_maraton lub próbki)
# df_data = df_maraton.sample(2000) 
df_data = dfx.sample(2000) # Uważaj na MemoryError dla hclust/ap na pełnych danych!
y_true_col = df_data['Kategoria wiekowa'] 

# Lista eksperymentów do wykonania (model, liczba klastrów,normalize, transform)
experiments = [
    ('kmeans', 5,False,False),
    ('kmeans', 10,False,False),
    ('birch', 5,False,False),
    ('birch', 10,False,False),
    ('ap', 5,False,False),
    ('ap', 10,False,False),
    ('meanshift', 5,False,False),
    ('meanshift', 10,False,False),
    ('kmeans', 5,True,False),
    ('kmeans', 10,True,False),
    ('birch', 5,True,False),
    ('birch', 10,True,False),
    ('ap', 5,True,False),
    ('ap', 10,True,False),
     ('meanshift', 5,True,False),
    ('meanshift', 10,True,False),
    ('kmeans', 5,False,True),
    ('kmeans', 10,False,True),
    ('birch', 5,False,True),
    ('birch', 10,False,True),
    ('ap', 5,False,True),
    ('ap', 10,False,True),
    ('meanshift', 5,False,True),
    ('meanshift', 10,False,True),
    ('kmeans', 5,True,True),
    ('kmeans', 10,True,True),
    ('birch', 5,True,True),
    ('birch', 10,True,True),
    ('ap', 5,True,True),
    ('ap', 10,True,True),
    ('meanshift', 5,True,True),
    ('meanshift', 10,True,True),

]

for model_type, k_value,normalize,transform in experiments:
    # Generujemy unikalną nazwę dla eksperymentu
    exp_name = f"{model_type.upper()}_k{k_value}_n_{normalize}_t_{transform}"
    
    print(f"Uruchamiam eksperyment: {exp_name}")
    
    # Trenujemy model i odbieramy 2 wartości
    model, metrics_df = build_model(model_type, df_data, y_true_col,num_clusters=k_value, normalize=normalize, transform=transform)
    
    # Przechowujemy pełny wynik w słowniku
    results_storage[exp_name] = (model, metrics_df)
    
    # Dodajemy metryki do listy do późniejszego porównania
    all_metrics_list.append(metrics_df)




Uruchamiam eksperyment: KMEANS_k5_n_False_t_False
Uruchamiam eksperyment: KMEANS_k10_n_False_t_False
Uruchamiam eksperyment: BIRCH_k5_n_False_t_False
Uruchamiam eksperyment: BIRCH_k10_n_False_t_False
Uruchamiam eksperyment: AP_k5_n_False_t_False
Uruchamiam eksperyment: AP_k10_n_False_t_False
Uruchamiam eksperyment: MEANSHIFT_k5_n_False_t_False
Uruchamiam eksperyment: MEANSHIFT_k10_n_False_t_False
Uruchamiam eksperyment: KMEANS_k5_n_True_t_False
Uruchamiam eksperyment: KMEANS_k10_n_True_t_False
Uruchamiam eksperyment: BIRCH_k5_n_True_t_False
Uruchamiam eksperyment: BIRCH_k10_n_True_t_False
Uruchamiam eksperyment: AP_k5_n_True_t_False
Uruchamiam eksperyment: AP_k10_n_True_t_False
Uruchamiam eksperyment: MEANSHIFT_k5_n_True_t_False
Uruchamiam eksperyment: MEANSHIFT_k10_n_True_t_False
Uruchamiam eksperyment: KMEANS_k5_n_False_t_True
Uruchamiam eksperyment: KMEANS_k10_n_False_t_True
Uruchamiam eksperyment: BIRCH_k5_n_False_t_True
Uruchamiam eksperyment: BIRCH_k10_n_False_t_True
Uruchamiam e

# Analiza Porównawcza i Wybór Modelu

W celu wyłonienia optymalnego algorytmu, wyniki wszystkich modeli zostały zestawione w tabeli zbiorczej. Kluczowymi wskaźnikami, na których oparto decyzję o wyborze modelu produkcyjnego, są:

    Silhouette Score (Podświetlony):
        Co oznacza: Mierzy, jak blisko swojego klastra znajduje się każdy punkt w porównaniu do klastrów sąsiednich.
        Interpretacja: Wysoka wartość Silhouette wskazuje, że klastry są dobrze odseparowane i nie nakładają się na siebie. Jest to kluczowe dla LLM – wyraźne różnice w danych wejściowych pozwalają na wygenerowanie unikalnych i trafnych opisów dla każdej grupy.
    Homogeneity (Podświetlony):
        Co oznacza: Sprawdza, czy klastry są spójne pod kątem cech kategorycznych (np. płeć, kategorie wiekowe).
        Interpretacja: Wysoka jednorodność oznacza, że podział wygenerowany przez algorytm "pokrywa się" z naturalnymi podziałami społecznymi biegaczy. Dzięki temu nazwy klastrów (np. "Szybkie Kobiety") są osadzone w rzeczywistości, a nie są jedynie abstrakcyjnym zbiorem liczb.

Wniosek: Model z najwyższymi wartościami w tych polach zapewnia najlepszy balans między matematyczną poprawnością a możliwością interpretacji wyników przez człowieka.

In [38]:
# --- Analiza wyników ---

# 1. Zbiorcza tabela metryk (do szybkiego porównania)
df_comparison = pd.concat(all_metrics_list, ignore_index=True)
print("\n--- Tabela Porównawcza Metryk ---")


# Funkcja pomocnicza do "inteligentnego" kolorowania rozmiaru klastra
def color_min_pct(val):
    if 2.0 <= val <= 15.0:
        return 'background-color: #c7e9c0; color: black' # Jasnozielony - idealnie
    elif val < 2.0:
        return 'background-color: #ff9999; color: black' # Czerwony - za małe!
    return '' # Reszta neutralna

# Budujemy Stylera
df_final_styled = df_comparison.style.applymap(
    color_min_pct, subset=['Min_Cluster_Pct']
).background_gradient(
    subset=['Homogeneity', 'Silhouette'], 
    cmap='YlGn' # Żółto-zielony gradient dla jakości
).background_gradient(
    subset=['Max_Cluster_Size'], 
    cmap='YlOrRd' # Żółto-pomarańczowy-czerwony (im większy klaster, tym bardziej ostrzegawczo)
).applymap(
    lambda x: 'background-color: #74c476; font-weight: bold' if 5 <= x <= 12 else '',
    subset=['n_clusters'] # Zielony dla Twojej liczby opisów w JSON
).format({
    'Min_Cluster_Pct': '{:.2f}%',
    'Homogeneity': '{:.3f}',
    'Silhouette': '{:.3f}'
})

# Przygotowanie ostatecznej, stylowej tabeli
df_final_styled = df_comparison.style \
    .background_gradient(subset=['Homogeneity', 'Silhouette'], cmap='YlGn') \
    .applymap(color_min_pct, subset=['Min_Cluster_Pct']) \
    .hide() \
    .set_properties(subset=['model'], **{'font-weight': 'bold', 'text-align': 'left'}) \
    .highlight_max(subset=['Homogeneity'], props='font-weight: bold; color: white; background-color: #2e7d32;') \
    .format(precision=3)


# Wyświetlenie w Notebooku
show(df_final_styled, classes="display nowrap cell-border") # itables pozwala klikać w nagłówki!



--- Tabela Porównawcza Metryk ---


model,Silhouette,Calinski-Harabasz,Davies-Bouldin,Homogeneity,Rand Index,Completeness,Min_Pct,Precision,Recall,F1_Score,Min_Cluster_Size,Min_Cluster_Pct,Max_Cluster_Size,n_clusters
KMEANS_k5_n_False_t_False,0.426,5440.113,0.705,0.031,0,0.047,3.600,0.029,0.067,0.032,72,3.600,703,5
KMEANS_k10_n_False_t_False,0.300,3933.794,1.018,0.071,0,0.070,0.250,0.164,0.122,0.127,5,0.250,388,10
BIRCH_k5_n_False_t_False,0.359,4766.684,0.776,0.043,0,0.064,3.600,0.031,0.079,0.038,72,3.600,674,5
BIRCH_k10_n_False_t_False,0.290,3865.661,0.905,0.066,0,0.067,0.250,0.096,0.070,0.055,5,0.250,438,10
AP_k5_n_False_t_False,0.273,2827.904,1.010,0.310,0,0.171,0.100,0.054,0.012,0.019,2,0.100,109,50
AP_k10_n_False_t_False,0.273,2827.904,1.010,0.310,0,0.171,0.100,0.054,0.012,0.019,2,0.100,109,50
MEANSHIFT_k5_n_False_t_False,0.475,2349.375,0.677,0.008,0,0.029,0.250,0.034,0.078,0.035,5,0.250,1621,4
MEANSHIFT_k10_n_False_t_False,0.475,2349.375,0.677,0.008,0,0.029,0.250,0.034,0.078,0.035,5,0.250,1621,4
KMEANS_k5_n_True_t_False,0.245,845.768,1.417,0.236,0,0.307,12.900,0.018,0.050,0.026,258,12.900,519,5
KMEANS_k10_n_True_t_False,0.206,558.071,1.591,0.293,0,0.271,4.150,0.171,0.105,0.127,83,4.150,400,10


# Selekcja Modelu Produkcyjnego (Model Selection Logic)

Proces wyboru ostatecznego modelu nie opiera się wyłącznie na najwyższych metrykach, ale przede wszystkim na kryteriach stabilności operacyjnej. Zastosowano wielostopniowy filtr bezpieczeństwa:

    Kompatybilność API: Ograniczono wybór do modeli K-Means i BIRCH, które natywnie wspierają metodę .predict(). Jest to niezbędne do poprawnego działania aplikacji w trybie "na żywo" na DigitalOcean.
    Reprezentatywność grup (Min_Cluster_Pct >= 2%): Wyeliminowano modele tworzące zbyt małe klastry (tzw. outliery), które nie niosą wartości statystycznej i mogłyby mylić model językowy LLM.
    Zdolność do uogólniania (Max_Cluster_Size < 80%): Odrzucono modele, które "idą na łatwiznę", wrzucając większość biegaczy do jednego ogromnego worka. Dążymy do realnego podziału społeczności.
    Złożoność (n_clusters <= 15): Ustalono limit grup, aby zachować czytelność raportu i zmieścić się w limitach kontekstu (Context Window) API OpenAI.

Ostateczny wybór: Spośród modeli spełniających powyższe rygory, wybrano ten o najwyższym wskaźniku Homogeneity, co gwarantuje, że podział matematyczny najlepiej pokrywa się z rzeczywistymi cechami biegaczy.

In [39]:
# Łączymy wyniki
df_results = pd.concat(all_metrics_list)

# FILTRUJEMY: 
# 1. Tylko modele z metodą predict (KMEANS, BIRCH)
# 2. Minimum 2% danych w najmniejszym klastrze (żadnych 'samotników')
# 3. Maksymalnie 15 klastrów (żeby zmieściły się w JSON)
# 4. Największy klaster nie może mieć więcej niż 80% danych (żeby model faktycznie dzielił)

df_safe = df_results[
    (df_results['model'].str.contains('KMEANS|BIRCH')) & 
    (df_results['Min_Cluster_Pct'] >= 2.0) & 
    (df_results['n_clusters'] <= 15) &
    (df_results['Max_Cluster_Size'] < (len(df_data) * 0.8))
]

# Z tych bezpiecznych wybieramy ten z najwyższym Homogeneity
if not df_safe.empty:
    best_safe_model_name = df_safe.sort_values(by='Homogeneity', ascending=False).iloc[0]['model']
    print(f"Najlepszy model do APLIKACJI: {best_safe_model_name}")
else:
    print("Brak modelu spełniającego kryteria bezpieczeństwa. Spróbuj zwiększyć k w K-Means lub włączyć normalizację.")

Najlepszy model do APLIKACJI: BIRCH_k10_n_False_t_True


# Zapis wybranego modelu

Na koniec analizy zapisujemy model lokalnie i w digital ocean.

In [40]:
# 1. Pobieramy nazwę z tabeli
best_name = str(best_safe_model_name).strip().lower()

# 2. Szukamy pasującego klucza w słowniku
actual_key = None
for key in results_storage.keys():
    if key.strip().lower() == best_name:
        actual_key = key
        break

if actual_key:
    # Wyciągamy model (pierwszy element pary)
    best_model_object = results_storage[actual_key][0]
    
    # 3. Zapisujemy
    from pycaret.clustering import save_model
    nazwa_pliku = "model_maraton"
    sciezka_pliku = data_dir / nazwa_pliku
    save_model(best_model_object, sciezka_pliku)
    print(f"Sukces! Model zapisany lokalnie pod kluczem: {actual_key}")
    #print(f"Zawartość modelu: {best_model_object}")
    do_access_key, do_secret_key, do_space_input, do_region, do_end_url = handle_digital_ocean_keys()
    upload_model_to_digital_ocean(str(sciezka_pliku) + ".pkl",do_access_key, do_secret_key, do_space_input,None,do_region, do_end_url)
else:
    print(f"Błąd: Nie znaleziono klucza '{best_name}' w słowniku.")
    print(f"Dostępne klucze to: {list(results_storage.keys())[:5]}...")

Transformation Pipeline and Model Successfully Saved
Sukces! Model zapisany lokalnie pod kluczem: BIRCH_k10_n_False_t_True
Model e:\od_zera_do_ai\pyt\zad9_1\data\model_maraton.pkl został wysłany do Space: maraton jako model_maraton.pkl


In [ ]:
import os
import re

# 1. Konfiguracja nazw
notebook_name = "notebook.ipynb"
html_name = notebook_name.replace(".ipynb", ".slides.html")

# 2. Czysta konwersja (BEZ --post serve!)
print("🏗️ Budowanie czystego pliku HTML...")
os.system(f'jupyter nbconvert "{notebook_name}" --to slides --no-input')

# 3. Naprawa skryptu sterującego
if os.path.exists(html_name):
    with open(html_name, "r", encoding="utf-8") as f:
        html_code = f.read()

    # Ta mała sztuczka dopisuje komendę wymuszającą auto-slajdy na samym końcu body
    # Działa nawet jeśli główna konfiguracja Reveal.js jest zablokowana
    patch = """
    <script>
    setTimeout(function() {
        if (window.Reveal) {
            Reveal.configure({ 
                autoSlide: 5000, 
                loop: true, 
                transition: 'convex',
                hash: true 
            });
            console.log("Auto-przewijanie aktywowane!");
        }
    }, 1000);
    </script>
    </body>
    """
    
    # Zamieniamy zamknięcie body na nasz patch
    new_html = html_code.replace("</body>", patch)

    with open(html_name, "w", encoding="utf-8") as f:
        f.write(new_html)
    
    print(f"✅ Sukces! Plik '{html_name}' został naprawiony.")
    print("👉 Teraz otwórz ten plik RĘCZNIE w przeglądarce (dwuklik na plik).")
else:
    print("❌ Nie znaleziono pliku HTML. Sprawdź czy nazwa notebooka jest poprawna.")


In [ ]:
import os

# Komenda zapisana jako tekst, żeby nic jej nie zepsuło
command = 'python -m jupyter nbconvert --to html --no-input notebook.ipynb'

# Uruchomienie i zignorowanie wyjścia
os.system(f'{command} >nul 2>&1')